Performance 🏃 
---
https://github.com/jeffmur/fhe-video-similarity/wiki/experiment

This notebook serves to analyze, compare, and visulize the trade-off of performance when using Fully Homomorphic Encryption (FHE) to compute video similarity scores on mobile vs. desktop devices.

Note: The multi-threading feature for pre-processing is disabled for all experiments, as we value consistency over performance.

Note: Every experiment uses the same encryption scheme & parameters:

* Cryptosystem: CKKS
* Polynomial Degree: 4096
* Encode Scalar: 2^40
* qSizes: [60, 40, 40, 60]

There are three metrics being gather within the application:

⚙️ **Pre-processing Time**: The time it takes to convert the video into a format that can be used for comparison.

📊 **Similarity Scores**: The time it takes to encrypt & compute a similarity score.

# Experiment 1: On-device Comparison

In this experiment, we compare the pre-processing and encryption duration on mobile vs. desktop devices.

We aim to learn how the performance of the application varies amongst devices (mobile vs. desktop). The scenarios are split by resolution (720p, 1080p, 2160p) and the video length is a constant 60 seconds. We select 60 seconds, as there is a direct comparison between the devices, as well as baseline comparison to Pop-Share.

| Device | OS
| --- | --- |
| Samsung S9 | Android
| Pixel 3XL | Android
| PC | Linux
| Raspberry Pi 400 | Linux

In [1]:
from utils.performance_tables import *
from utils import TARGET_SYS, FRAME_COUNTS
# To be summarized in table
kld_err = []
bhatt_err = []
cram_err = []

def add_mean_error(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    kld, bhattacharyya, cramer = mean_error(pathToAssertion, os, frameCounts).values()
    kld_err.append(kld)
    bhatt_err.append(bhattacharyya)
    cram_err.append(cramer)

def insert_or_append_float(old:dict[str,list[float]], new:dict[str,float]):
    for k, v in new.items():
        if k in old:
            old[k].append(v)
        else:
            old[k] = [v]

def insert_or_append_dict(old:dict[str,dict[str,list[float]]], new:dict[str,float]):
    for k, v in new.items():
        if k in old:
            for k2, v2 in v.items():
                if k2 in old[k]:
                    old[k][k2].append(v2)
                else:
                    old[k][k2] = [v2]
        else:
            old[k] = {k2: v2 if isinstance(v2, list) else [v2] for k2, v2 in v.items()}

# Aggregated Pre-processing durations
pp_by_sys = {}
pp_by_res = {}

# Operations FHE vs. Plaintext
ops_by_sys = {}

def add_metric(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    # OS pre-processing
    sys = pre_processing_by_sys(pathToAssertion, os, frameCounts)
    insert_or_append_float(pp_by_sys, sys)
    
    # Resolution pre-processing
    res = pre_processing_by_res(pathToAssertion.split('/')[1], pathToAssertion, os, frameCounts)
    insert_or_append_float(pp_by_res, res)
    
    # Mean Error FHE vs. Plaintext
    add_mean_error(pathToAssertion, os, frameCounts)

    # Operations FHE vs. Plaintext
    ops = operations_by_sys(pathToAssertion, os, frameCounts)
    print(ops_by_sys)
    insert_or_append_dict(ops_by_sys, ops)
    print(ops_by_sys)

## Scenario 1: 720p

On every device, import and pre-process the video twice, compare against itself, using two different keys.


### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1280x720 -c:v libx264 -t 60 -an Black_720p_60s.mp4
```

In [2]:
black_720p = "3_performance/720p/60s_Black"
add_metric(black_720p)
verbose_md_table(black_720p)

{}
{'pc': {'encryption': [370.0], 'fhe_compute': [190.0], 'pt_compute': [1.141]}, 's9': {'encryption': [5095.0], 'fhe_compute': [2593.0], 'pt_compute': [2.153]}, 'pxl': {'encryption': [2998.0], 'fhe_compute': [1559.0], 'pt_compute': [3.094]}}


Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -3.06e-11 [100.00%] | 6.12e-11 | 13.00 | 157.00 | 48.00 | 0.73 | 47.27 [6484.36%]
Cramer [pc] [all] | 1.31e-09 [100.00%] | 2.62e-09 | 13.00 | 71.00 | 113.00 | 0.21 | 112.79 [53201.89%]
BC [pc] [all] | 1.00e+00 [100.00%] | 3.49e-10 | 13.00 | 142.00 | 29.00 | 0.20 | 28.80 [14400.00%]
KLD [s9] [all] | 3.12e-11 [100.00%] | 4.86e-11 | 184.00 | 1891.00 | 663.00 | 1.00 | 662.00 [66200.00%]
Cramer [s9] [all] | 1.44e-09 [100.00%] | 1.26e-09 | 184.00 | 986.00 | 1567.00 | 0.64 | 1566.36 [242845.74%]
BC [s9] [all] | 1.00e+00 [100.00%] | 6.38e-10 | 184.00 | 2218.00 | 363.00 | 0.51 | 362.49 [71356.69%]
KLD [pxl] [all] | 6.54e-07 [100.00%] | 1.23e-11 | 199.50 | 1216.00 | 384.00 | 2.00 | 382.00 [19100.00%]
Cramer [pxl] [all] | 4.80e-04 [100.00%] | 1.22e-10 | 199.50 | 596.00 | 951.00 | 0.71 | 950.29 [133280.08%]
BC [pxl] [all] | 1.00e+00 [99.95%] | 4.77e-10 | 199.50 | 1186.00 | 224.00 | 0.38 | 223.62 [58692.65%]

### Test 2: Samsung S9

Taken from Handheld Samsung S9, 720p, 60 seconds.

In [3]:
s9_720p = "3_performance/720p/60s_S9"
add_metric(s9_720p)
verbose_md_table(s9_720p)

{'pc': {'encryption': [370.0], 'fhe_compute': [190.0], 'pt_compute': [1.141]}, 's9': {'encryption': [5095.0], 'fhe_compute': [2593.0], 'pt_compute': [2.153]}, 'pxl': {'encryption': [2998.0], 'fhe_compute': [1559.0], 'pt_compute': [3.094]}}
{'pc': {'encryption': [370.0, 364.0], 'fhe_compute': [190.0, 192.0], 'pt_compute': [1.141, 0.9810000000000001]}, 's9': {'encryption': [5095.0, 1531.0], 'fhe_compute': [2593.0, 822.0], 'pt_compute': [2.153, 1.739]}, 'pxl': {'encryption': [2998.0, 2034.0], 'fhe_compute': [1559.0, 1062.0], 'pt_compute': [3.094, 0.106]}}


Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -1.97e-12 [100.00%] | 3.94e-12 | 71.00 | 152.00 | 48.00 | 0.61 | 47.39 [7730.34%]
Cramer [pc] [all] | 7.49e-10 [100.00%] | 1.50e-09 | 71.00 | 71.00 | 114.00 | 0.21 | 113.79 [54972.46%]
BC [pc] [all] | 1.00e+00 [100.00%] | 2.42e-10 | 71.00 | 141.00 | 30.00 | 0.16 | 29.84 [18533.54%]
KLD [s9] [all] | 6.61e-12 [100.00%] | 1.32e-11 | 442.50 | 627.00 | 209.00 | 0.77 | 208.23 [27184.60%]
Cramer [s9] [all] | 8.44e-10 [100.00%] | 1.69e-09 | 442.50 | 303.00 | 496.00 | 0.69 | 495.31 [71993.02%]
BC [s9] [all] | 1.00e+00 [100.00%] | 3.65e-10 | 442.50 | 601.00 | 117.00 | 0.28 | 116.72 [40952.63%]
KLD [pxl] [all] | 3.95e-11 [100.00%] | 7.90e-11 | 454.00 | 823.00 | 261.00 | 0.04 | 260.96 [705305.41%]
Cramer [pxl] [all] | 4.52e-10 [100.00%] | 9.03e-10 | 454.00 | 405.00 | 651.00 | 0.05 | 650.95 [1415117.39%]
BC [pxl] [all] | 1.00e+00 [100.00%] | 2.17e-10 | 454.00 | 806.00 | 150.00 | 0.02 | 149.98 [652073.91%]

## Scenario 2: 1080p



### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1920x1080 -c:v libx264 -t 60 -an Black_1080p_60s.mp4
```

In [4]:
black_1080p = "3_performance/1080p/60s_Black"
add_metric(black_1080p)
verbose_md_table(black_1080p)

{'pc': {'encryption': [370.0, 364.0], 'fhe_compute': [190.0, 192.0], 'pt_compute': [1.141, 0.9810000000000001]}, 's9': {'encryption': [5095.0, 1531.0], 'fhe_compute': [2593.0, 822.0], 'pt_compute': [2.153, 1.739]}, 'pxl': {'encryption': [2998.0, 2034.0], 'fhe_compute': [1559.0, 1062.0], 'pt_compute': [3.094, 0.106]}}
{'pc': {'encryption': [370.0, 364.0], 'fhe_compute': [190.0, 192.0], 'pt_compute': [1.141, 0.9810000000000001]}, 's9': {'encryption': [5095.0, 1531.0, 2503.0], 'fhe_compute': [2593.0, 822.0, 1299.0], 'pt_compute': [2.153, 1.739, 2.257]}, 'pxl': {'encryption': [2998.0, 2034.0, 5956.0], 'fhe_compute': [1559.0, 1062.0, 3068.0], 'pt_compute': [3.094, 0.106, 2.171]}}


Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [s9] [all] | 4.62e-11 [100.00%] | 9.24e-11 | 338.00 | 987.00 | 322.00 | 1.00 | 321.00 [32100.00%]
Cramer [s9] [all] | 1.47e-09 [100.00%] | 2.93e-09 | 338.00 | 499.00 | 788.00 | 0.82 | 787.17 [95415.15%]
BC [s9] [all] | 1.00e+00 [100.00%] | 1.56e-11 | 338.00 | 1017.00 | 189.00 | 0.43 | 188.57 [43650.00%]
KLD [pxl] [all] | 6.48e-11 [100.00%] | 1.29e-10 | 366.00 | 2398.00 | 744.00 | 1.00 | 743.00 [74300.00%]
Cramer [pxl] [all] | 1.57e-09 [100.00%] | 1.72e-09 | 366.00 | 1187.00 | 1887.00 | 0.76 | 1886.23 [246566.67%]
BC [pxl] [all] | 1.00e+00 [100.00%] | 8.90e-10 | 366.00 | 2371.00 | 437.00 | 0.41 | 436.59 [107535.47%]

### Test 2: Pixel 3XL

Taken from Handheld Pixel 3XL, 1080p, 60 seconds.

In [5]:
pxl_1080p = "3_performance/1080p/60s_PXL"
add_metric(pxl_1080p)
verbose_md_table(pxl_1080p)

{'pc': {'encryption': [370.0, 364.0], 'fhe_compute': [190.0, 192.0], 'pt_compute': [1.141, 0.9810000000000001]}, 's9': {'encryption': [5095.0, 1531.0, 2503.0], 'fhe_compute': [2593.0, 822.0, 1299.0], 'pt_compute': [2.153, 1.739, 2.257]}, 'pxl': {'encryption': [2998.0, 2034.0, 5956.0], 'fhe_compute': [1559.0, 1062.0, 3068.0], 'pt_compute': [3.094, 0.106, 2.171]}}
{'pc': {'encryption': [370.0, 364.0, 357.0], 'fhe_compute': [190.0, 192.0, 187.0], 'pt_compute': [1.141, 0.9810000000000001, 0.55]}, 's9': {'encryption': [5095.0, 1531.0, 2503.0, 2414.0], 'fhe_compute': [2593.0, 822.0, 1299.0, 1253.0], 'pt_compute': [2.153, 1.739, 2.257, 2.364]}, 'pxl': {'encryption': [2998.0, 2034.0, 5956.0, 2414.0], 'fhe_compute': [1559.0, 1062.0, 3068.0, 1280.0], 'pt_compute': [3.094, 0.106, 2.171, 1.968]}}


Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | 1.56e-11 [100.00%] | 3.12e-11 | 310.00 | 144.00 | 45.00 | 0.37 | 44.63 [12062.16%]
Cramer [pc] [all] | 4.03e-10 [100.00%] | 8.06e-10 | 310.00 | 71.00 | 113.00 | 0.09 | 112.91 [124075.82%]
BC [pc] [all] | 1.00e+00 [100.00%] | 1.12e-10 | 310.00 | 142.00 | 29.00 | 0.09 | 28.91 [32484.27%]
KLD [s9] [all] | 3.36e-11 [100.00%] | 6.71e-11 | 728.50 | 985.00 | 318.00 | 1.00 | 317.00 [31700.00%]
Cramer [s9] [all] | 2.25e-10 [100.00%] | 4.49e-10 | 728.50 | 465.00 | 758.00 | 1.00 | 757.00 [75700.00%]
BC [s9] [all] | 1.00e+00 [100.00%] | 5.40e-10 | 728.50 | 964.00 | 177.00 | 0.36 | 176.64 [48526.37%]
KLD [pxl] [all] | 2.52e-08 [100.00%] | 2.46e-11 | 827.00 | 975.00 | 331.00 | 1.00 | 330.00 [33000.00%]
Cramer [pxl] [all] | 1.14e-04 [100.00%] | 1.11e-10 | 827.00 | 480.00 | 768.00 | 0.62 | 767.38 [124171.84%]
BC [pxl] [all] | 1.00e+00 [99.99%] | 4.00e-10 | 827.00 | 959.00 | 181.00 | 0.35 | 180.65 [51614.29%]

# Experiment 2: FHE vs. Plaintext Operations

In this experiment, we compare the performance of FHE vs. plaintext operations.

## Scenario 1: Absolute Mean Error

Aggregated from the previous experiments, the absolute mean error will be calculated to determine the accuracy of the similarity scores using FHE library.

In [6]:
mean_error_md_table(kld_err, bhatt_err, cram_err)

Function | Mean Error
---|---
KLD | 8.02e-12
Cramer | 1.23e-09
BC | 1.76e-10

## Scenario 2: Android vs. Linux


In [7]:
ops_by_sys

{'pc': {'encryption': [370.0, 364.0, 357.0],
  'fhe_compute': [190.0, 192.0, 187.0],
  'pt_compute': [1.141, 0.9810000000000001, 0.55]},
 's9': {'encryption': [5095.0, 1531.0, 2503.0, 2414.0],
  'fhe_compute': [2593.0, 822.0, 1299.0, 1253.0],
  'pt_compute': [2.153, 1.739, 2.257, 2.364]},
 'pxl': {'encryption': [2998.0, 2034.0, 5956.0, 2414.0],
  'fhe_compute': [1559.0, 1062.0, 3068.0, 1280.0],
  'pt_compute': [3.094, 0.106, 2.171, 1.968]}}

In [8]:
operations_by_sys_md_table(ops_by_sys)

System | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms)
---|---|---|---
pc | 363.67 | 189.67 | 0.89
s9 | 2885.75 | 1491.75 | 2.13
pxl | 3350.50 | 1742.25 | 1.83

# Experiment 4: Pre-processing


## Scenario 1: Android vs. Linux

In [9]:
pp_by_sys

{'pc': [13.0, 71.0, 310.0],
 's9': [184.0, 442.5, 338.0, 728.5],
 'pxl': [199.5, 454.0, 366.0, 827.0]}

In [10]:
pre_processing_by_sys_md_table(pp_by_sys)

System | Average (s) | Min (s) | Max (s)
---|---|---|---
pc | 131.33 | 13.00 | 310.00
s9 | 423.25 | 184.00 | 728.50
pxl | 461.62 | 199.50 | 827.00

## Scenario 2: Resolution

In [11]:
pp_by_res

{'720p': [132.16666666666666, 322.5],
 '1080p': [234.66666666666666, 621.8333333333334]}

In [12]:
pre_processing_by_res_md_table(pp_by_res)

Resolution | Average (s) | Min (s) | Max (s)
---|---|---|---
720p | 227.33 | 132.17 | 322.50
1080p | 428.25 | 234.67 | 621.83